# Colab: `top-papers-graph` + fixed Task 1/Task 2 CSV metadata + Hugging Face

Этот блокнот делает полный цикл:

1. клонирует `https://github.com/top-papers/top-papers-graph`
2. устанавливает зависимости
3. скачивает **только два CSV metadata** из Google Sheets:
   - Task 1
   - Task 2
4. по данным этих CSV выбирает для каждого пользователя **самую позднюю** отправку по `Timestamp`
5. скачивает файлы решений **только из ссылок, которые лежат в этих CSV**
6. собирает curated input
7. запускает `top-papers-graph export-scidatapipe`
8. загружает результат в Hugging Face dataset repo

## Важно
- Блокнот больше **не использует links.txt**.
- Источником истины теперь являются только два metadata CSV.
- Upload в Hugging Face выполняется только если `sft.jsonl` и/или `grpo.jsonl` реально содержат данные.

- `MAX_IMAGES_PER_SAMPLE` управляет реальными image attachments; `MAX_MULTIMODAL_RECORDS_PER_SAMPLE` управляет только текстовыми multimodal-summary records. Значение `0` в CLI означает «без лимита», но в этом блокноте выставлен конечный дефолт `8`, чтобы избежать путаницы и чрезмерно длинных prompt.


In [ ]:
# @title 1) Параметры

REPO_URL = "https://github.com/top-papers/top-papers-graph"  # @param {type:"string"}
REPO_BRANCH = "main"  # @param {type:"string"}

# Необязательно: overlay-архив с патчем поверх публичного repo, если upstream ещё не содержит export-scidatapipe
REPO_OVERLAY_GDRIVE_FILE_ID = ""  # @param {type:"string"}
REPO_OVERLAY_GDRIVE_URL = ""  # @param {type:"string"}

# Фиксированные Google Sheets metadata
TASK1_SHEET_URL = "https://docs.google.com/spreadsheets/d/1e08TtCg4RxNuBPa426YyUgU5Xydc9VQffxqA4o1x4Ac/edit?usp=sharing"  # @param {type:"string"}
TASK2_SHEET_URL = "https://docs.google.com/spreadsheets/d/1nI3wXKo7GMdql1GOn9kVcriWhvra0iF_QqeLpOksSmg/edit?usp=sharing"  # @param {type:"string"}

WORK_DIR = "/content/workdir"  # @param {type:"string"}
CURATED_INPUT_DIR = "/content/curated_input"  # @param {type:"string"}
VALIDATED_INPUT_DIR = "/content/validated_input"  # @param {type:"string"}
OUTPUT_DIR = "/content/top-papers-graph/data/derived/scidatapipe_export"  # @param {type:"string"}

# Параметры bridge
COPY_ASSETS = True  # @param {type:"boolean"}
MAX_IMAGES_PER_SAMPLE = 8  # @param {type:"integer"}
# 0 в CLI означает "все текстовые multimodal records"; картинки контролируются MAX_IMAGES_PER_SAMPLE.
MAX_MULTIMODAL_RECORDS_PER_SAMPLE = 8  # @param {type:"integer"}
RECURSIVE = True  # @param {type:"boolean"}

DOWNLOAD_PAPERS = True  # @param {type:"boolean"}
DOWNLOAD_ROOT = "/content/download_cache"  # @param {type:"string"}
DOWNLOADED_PROCESSED_PAPERS_DIR = "/content/downloaded_processed_papers"  # @param {type:"string"}
UNPAYWALL_EMAIL = ""  # @param {type:"string"}
DOWNLOAD_MULTIMODAL = True  # @param {type:"boolean"}
DOWNLOAD_RUN_VLM = True  # @param {type:"boolean"}

# Hugging Face
HF_UPLOAD = True  # @param {type:"boolean"}
HF_REPO_ID = "top-papers/top-papers-graph-experts-data"  # @param {type:"string"}
HF_PRIVATE = False  # @param {type:"boolean"}
HF_PATH_IN_REPO = "exports/colab-run-001"  # @param {type:"string"}
HF_TOKEN = ""  # @param {type:"string"}
HF_USE_NOTEBOOK_LOGIN = True  # @param {type:"boolean"}
HF_COMMIT_MESSAGE = "Upload scidatapipe export from Colab"  # @param {type:"string"}
HF_COMMIT_DESCRIPTION = "Generated from Task1/Task2 metadata CSVs in Google Colab"  # @param {type:"string"}

print("Параметры загружены.")


## Установка зависимостей и клонирование репозитория

Для загрузки результата в Hub блокнот использует `upload_folder()` / dataset repo сценарий из `huggingface_hub`. ([huggingface.co](https://huggingface.co/docs/huggingface_hub/guides/upload))


In [ ]:
# @title
import os
import sys
import shutil
import subprocess
from pathlib import Path

def run(cmd, cwd=None):
    print("+", " ".join(cmd))
    subprocess.run(cmd, check=True, cwd=cwd)

run(["apt-get", "update", "-qq"])
run(["apt-get", "install", "-y", "-qq", "git", "p7zip-full", "unzip"])

run([sys.executable, "-m", "pip", "install", "-q", "-U", "pip", "setuptools", "wheel"])
run([sys.executable, "-m", "pip", "install", "-q", "-U",
     "gdown", "huggingface_hub", "pandas", "openpyxl", "PyYAML", "Unidecode", "requests"])

repo_dir = Path("/content/top-papers-graph")
if repo_dir.exists():
    shutil.rmtree(repo_dir)

run(["git", "clone", "--depth", "1", "--branch", REPO_BRANCH, REPO_URL, str(repo_dir)])
run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], cwd=str(repo_dir))

print("Репозиторий установлен:", repo_dir)


## Опционально: overlay-архив поверх публичного репозитория


In [ ]:
# @title
import re
import gdown
import zipfile
from pathlib import Path

def _extract_gdrive_file_id(url: str):
    if not url:
        return None
    patterns = [
        r"/file/d/([a-zA-Z0-9_-]+)",
        r"[?&]id=([a-zA-Z0-9_-]+)",
    ]
    for pattern in patterns:
        m = re.search(pattern, url)
        if m:
            return m.group(1)
    return None

def _gdown_download_compat(*, url=None, id=None, output=None, quiet=False):
    if id is not None:
        try:
            return gdown.download(id=id, output=output, quiet=quiet)
        except TypeError:
            direct_url = f"https://drive.google.com/uc?id={id}"
            try:
                return gdown.download(url=direct_url, output=output, quiet=quiet)
            except TypeError:
                return gdown.download(direct_url, output, quiet=quiet)

    if url is None:
        raise ValueError("Either url or id must be provided")

    try:
        return gdown.download(url=url, output=output, quiet=quiet)
    except TypeError:
        pass
    except Exception:
        pass

    try:
        return gdown.download(url, output, quiet=quiet)
    except Exception:
        pass

    file_id = _extract_gdrive_file_id(url)
    if file_id:
        direct_url = f"https://drive.google.com/uc?id={file_id}"
        try:
            return gdown.download(url=direct_url, output=output, quiet=quiet)
        except TypeError:
            return gdown.download(direct_url, output, quiet=quiet)

    return gdown.download(url, output, quiet=quiet)

def download_with_gdown(file_id: str, url: str, output_path: Path):
    if file_id.strip():
        return _gdown_download_compat(id=file_id.strip(), output=str(output_path), quiet=False)
    if url.strip():
        return _gdown_download_compat(url=url.strip(), output=str(output_path), quiet=False)
    return None

overlay_path = Path("/content/repo_overlay.zip")
overlay_downloaded = download_with_gdown(REPO_OVERLAY_GDRIVE_FILE_ID, REPO_OVERLAY_GDRIVE_URL, overlay_path)

if overlay_downloaded:
    print("Overlay скачан:", overlay_path)
    with zipfile.ZipFile(overlay_path, "r") as zf:
        zf.extractall(repo_dir.parent)

    extracted_dirs = [p for p in repo_dir.parent.iterdir() if p.is_dir() and p.name.startswith("top-papers-graph")]
    extracted_dirs = sorted(extracted_dirs, key=lambda p: len(p.parts), reverse=True)

    overlay_root = None
    for candidate in extracted_dirs:
        if candidate == repo_dir:
            continue
        if (candidate / "pyproject.toml").exists():
            overlay_root = candidate
            break

    if overlay_root is None:
        overlay_root = repo_dir.parent

    for src in overlay_root.rglob("*"):
        if src.is_dir():
            continue
        try:
            rel = src.relative_to(overlay_root)
        except ValueError:
            continue
        dst = repo_dir / rel
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(src, dst)

    run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], cwd=str(repo_dir))
    print("Overlay наложен.")
else:
    print("Overlay не задан — продолжаю с публичным репозиторием.")


## Скачивание только двух metadata CSV

Для Google Sheets блокнот переводит `edit`-URL в `export?format=csv`. Это исключает промежуточный `links.txt` и гарантирует, что все решения будут скачиваться только по ссылкам, записанным в этих таблицах.


In [ ]:
# @title
import os
import subprocess
from pathlib import Path

import pandas as pd
import gdown

work_dir = Path(WORK_DIR)
if work_dir.exists():
    shutil.rmtree(work_dir)
work_dir.mkdir(parents=True, exist_ok=True)

metadata_dir = work_dir / "metadata"
metadata_dir.mkdir(parents=True, exist_ok=True)

def download_csv(sheet_url: str, dst_path: Path):
    """
    Robust Google Sheets -> CSV downloader.
    1) Prefer gdown CLI with --format csv for share links.
    2) Fallback: download sheet via gdown Python API (usually xlsx) and convert to CSV.
    """
    dst_path.parent.mkdir(parents=True, exist_ok=True)

    # Try gdown CLI first because it handles Google Sheets export better than raw requests.
    cli_cmd = ["gdown", sheet_url, "--format", "csv", "-O", str(dst_path)]
    print("+", " ".join(cli_cmd))
    cli_result = subprocess.run(cli_cmd, capture_output=True, text=True)
    if cli_result.returncode == 0 and dst_path.exists() and dst_path.stat().st_size > 0:
        return dst_path

    print("gdown CLI csv export failed; falling back to Python API.")
    if cli_result.stdout:
        print("stdout:", cli_result.stdout[:2000])
    if cli_result.stderr:
        print("stderr:", cli_result.stderr[:2000])

    # Fallback: Python API download, then convert xlsx->csv if needed.
    tmp_dir = dst_path.parent / f"tmp_{dst_path.stem}"
    tmp_dir.mkdir(parents=True, exist_ok=True)

    before = {p.name for p in tmp_dir.iterdir()}
    result = None
    try:
        result = gdown.download(url=sheet_url, output=str(tmp_dir), quiet=False)
    except TypeError:
        result = gdown.download(sheet_url, str(tmp_dir), quiet=False)

    downloaded_files = [p for p in tmp_dir.iterdir() if p.is_file() and p.name not in before]
    if result and Path(result).exists():
        downloaded_files.append(Path(result))
    # unique, keep order
    uniq = []
    seen = set()
    for p in downloaded_files:
        if str(p) not in seen:
            seen.add(str(p))
            uniq.append(p)
    downloaded_files = uniq

    if not downloaded_files:
        raise RuntimeError(f"Failed to download spreadsheet: {sheet_url}")

    src = sorted(downloaded_files, key=lambda p: p.stat().st_mtime, reverse=True)[0]
    lower = src.name.lower()

    if lower.endswith(".csv"):
        dst_path.write_bytes(src.read_bytes())
        return dst_path

    if lower.endswith(".xlsx") or lower.endswith(".xls"):
        df = pd.read_excel(src)
        df.to_csv(dst_path, index=False)
        return dst_path

    raise RuntimeError(f"Downloaded spreadsheet has unsupported format: {src.name}")

task1_csv_path = metadata_dir / "task1.csv"
task2_csv_path = metadata_dir / "task2.csv"

download_csv(TASK1_SHEET_URL, task1_csv_path)
download_csv(TASK2_SHEET_URL, task2_csv_path)

df1 = pd.read_csv(task1_csv_path)
df2 = pd.read_csv(task2_csv_path)

print("Task1 CSV:", task1_csv_path)
print("Task2 CSV:", task2_csv_path)
print("Task1 columns:", list(df1.columns))
print("Task2 columns:", list(df2.columns))
print("\nTask1 head:")
print(df1.head(2))
print("\nTask2 head:")
print(df2.head(2))


## Дедупликация по `Timestamp` и скачивание файлов решений **только из CSV**


In [ ]:
# @title
import json
import re
import shutil
import zipfile
from pathlib import Path

import pandas as pd
import yaml
import gdown
from unidecode import unidecode

curated_dir = Path(CURATED_INPUT_DIR)
if curated_dir.exists():
    shutil.rmtree(curated_dir)
curated_dir.mkdir(parents=True, exist_ok=True)

downloads_dir = work_dir / "downloads"
downloads_dir.mkdir(parents=True, exist_ok=True)

def _extract_gdrive_file_id(url: str):
    if not url:
        return None
    patterns = [
        r"/file/d/([a-zA-Z0-9_-]+)",
        r"[?&]id=([a-zA-Z0-9_-]+)",
    ]
    for pattern in patterns:
        m = re.search(pattern, url)
        if m:
            return m.group(1)
    return None

def _gdown_download_compat(*, url=None, id=None, output=None, quiet=False):
    """
    Совместимый helper для разных версий gdown.
    Работает и для ссылок вида open?id=..., и для file/d/... .
    """
    if id is not None:
        direct_url = f"https://drive.google.com/uc?id={id}"
        try:
            return gdown.download(url=direct_url, output=output, quiet=quiet)
        except TypeError:
            return gdown.download(direct_url, output, quiet=quiet)

    if url is None:
        raise ValueError("Either url or id must be provided")

    # 1) keyword-style
    try:
        return gdown.download(url=url, output=output, quiet=quiet)
    except TypeError:
        pass
    except Exception:
        pass

    # 2) positional-style
    try:
        return gdown.download(url, output, quiet=quiet)
    except Exception:
        pass

    # 3) normalize share link -> uc?id=<file_id>
    file_id = _extract_gdrive_file_id(url)
    if file_id:
        direct_url = f"https://drive.google.com/uc?id={file_id}"
        try:
            return gdown.download(url=direct_url, output=output, quiet=quiet)
        except TypeError:
            return gdown.download(direct_url, output, quiet=quiet)

    # Let last attempt surface
    return gdown.download(url, output, quiet=quiet)

def parse_timestamp(ts):
    if pd.isna(ts):
        return pd.NaT
    return pd.to_datetime(str(ts), errors="coerce", utc=True)

def normalize_text(text: str) -> str:
    text = str(text or "").strip()
    text = text.replace("\u00A0", " ")
    text = unidecode(text).lower()
    text = re.sub(r"__+[0-9a-f]{6,}", "", text)
    text = re.sub(r"\s*-\s*[^/\\]+$", "", text)
    text = re.sub(r"[^a-z0-9]+", "_", text)
    text = re.sub(r"_+", "_", text).strip("_")
    return text

def file_person_key_from_name(filename: str) -> str:
    stem = Path(filename).stem
    stem = re.sub(r"__+[0-9a-f]{6,}", "", stem, flags=re.I)
    stem = re.sub(r"\s*-\s*[^/\\]+$", "", stem)
    return normalize_text(stem)

def split_urls(value):
    if pd.isna(value):
        return []
    text = str(value).strip()
    if not text:
        return []
    parts = re.split(r"[;\n\r,]+", text)
    return [p.strip() for p in parts if p.strip()]

def download_any_gdown(url: str, output_dir: Path):
    output_dir.mkdir(parents=True, exist_ok=True)
    before = {p.name for p in output_dir.iterdir()}
    try:
        result = _gdown_download_compat(url=url, output=str(output_dir), quiet=False)
    except Exception as e:
        print("gdown failed:", url, "->", repr(e))
        return None

    after_files = [p for p in output_dir.iterdir() if p.is_file()]
    new_files = [p for p in after_files if p.name not in before]
    if len(new_files) == 1:
        return new_files[0]
    if result and Path(result).exists():
        return Path(result)
    if after_files:
        after_files = sorted(after_files, key=lambda p: p.stat().st_mtime, reverse=True)
        return after_files[0]
    return None

def find_upload_col(df, task):
    cols = list(df.columns)
    if task == "task1":
        for c in cols:
            if "yaml" in str(c).lower():
                return c
    if task == "task2":
        for c in cols:
            lc = str(c).lower()
            if ".zip" in lc or "архив" in lc:
                return c
    return None

manifest = {
    "task1_metadata_csv": str(task1_csv_path),
    "task2_metadata_csv": str(task2_csv_path),
    "task1_selected": [],
    "task2_selected": [],
    "task1_failed_downloads": [],
    "task2_failed_downloads": [],
}

# Task 1
task1_upload_col = find_upload_col(df1, "task1")
if task1_upload_col is None:
    raise RuntimeError("Не удалось найти колонку с YAML ссылками в Task 1 CSV")

df1 = df1.copy()
if "Timestamp" not in df1.columns:
    raise RuntimeError("В Task 1 CSV отсутствует колонка Timestamp")
df1["__timestamp"] = df1["Timestamp"].apply(parse_timestamp)

task1_probe_dir = downloads_dir / "task1_probe"
task1_probe_dir.mkdir(parents=True, exist_ok=True)

task1_records = []
for _, row in df1.iterrows():
    for url in split_urls(row.get(task1_upload_col)):
        path = download_any_gdown(url, task1_probe_dir)
        if path is None:
            manifest["task1_failed_downloads"].append({"url": url})
            continue
        user_key = file_person_key_from_name(path.name)
        task1_records.append({
            "timestamp": row["__timestamp"],
            "url": url,
            "filename": path.name,
            "downloaded_path": str(path),
            "user_key": user_key,
        })

task1_latest = {}
for rec in task1_records:
    key = rec["user_key"]
    ts = rec["timestamp"]
    current = task1_latest.get(key)
    if current is None or (pd.notna(ts) and (pd.isna(current["timestamp"]) or ts > current["timestamp"])):
        task1_latest[key] = rec

for rec in task1_latest.values():
    src = Path(rec["downloaded_path"])
    dst = curated_dir / src.name
    shutil.copy2(src, dst)
    manifest["task1_selected"].append({
        "user_key": rec["user_key"],
        "timestamp": str(rec["timestamp"]),
        "url": rec["url"],
        "filename": rec["filename"],
    })

# Task 2
task2_upload_col = find_upload_col(df2, "task2")
if task2_upload_col is None:
    raise RuntimeError("Не удалось найти колонку с ZIP ссылками в Task 2 CSV")

df2 = df2.copy()
if "Timestamp" not in df2.columns:
    raise RuntimeError("В Task 2 CSV отсутствует колонка Timestamp")
df2["__timestamp"] = df2["Timestamp"].apply(parse_timestamp)

def task2_person_key(row):
    for col in ["ФИО", "Username", "Email Address"]:
        if col in row and pd.notna(row[col]) and str(row[col]).strip():
            return normalize_text(row[col])
    return ""

df2["__person_key"] = df2.apply(task2_person_key, axis=1)

task2_latest = {}
for _, row in df2.iterrows():
    key = row["__person_key"] or "unknown_user"
    ts = row["__timestamp"]
    current = task2_latest.get(key)
    if current is None or (pd.notna(ts) and (pd.isna(current["__timestamp"]) or ts > current["__timestamp"])):
        task2_latest[key] = row

for _, row in task2_latest.items():
    chosen_files = []
    for url in split_urls(row.get(task2_upload_col)):
        person_dir = downloads_dir / "task2" / (row["__person_key"] or "unknown")
        person_dir.mkdir(parents=True, exist_ok=True)
        path = download_any_gdown(url, person_dir)
        if path is None:
            manifest["task2_failed_downloads"].append({
                "user_key": row["__person_key"],
                "url": url,
            })
            continue
        dst = curated_dir / path.name
        shutil.copy2(path, dst)
        chosen_files.append(path.name)

    manifest["task2_selected"].append({
        "user_key": row["__person_key"],
        "timestamp": str(row["__timestamp"]),
        "downloaded_files": chosen_files,
    })

manifest_path = curated_dir / "curation_manifest.json"
manifest_path.write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8")

real_inputs = [p for p in curated_dir.iterdir() if p.is_file() and p.name != "curation_manifest.json"]
print("Selected Task1 files:", len(manifest["task1_selected"]))
print("Selected Task2 rows :", len(manifest["task2_selected"]))
print("Real curated inputs :", len(real_inputs))
for p in sorted(real_inputs)[:20]:
    print("-", p.name)

if not real_inputs:
    raise RuntimeError("После скачивания по CSV не найдено ни одного входного файла.")


## Лёгкая валидация входных файлов


In [ ]:
# @title
validated_input_dir = Path(VALIDATED_INPUT_DIR)
if validated_input_dir.exists():
    shutil.rmtree(validated_input_dir)
validated_input_dir.mkdir(parents=True, exist_ok=True)

validation_manifest = {
    "validation_mode": "lightweight",
    "accepted_task1": [],
    "accepted_task2": [],
    "rejected_task1": [],
    "rejected_task2": [],
}

def lightweight_validate_yaml(path: Path):
    with path.open("r", encoding="utf-8") as f:
        obj = yaml.safe_load(f)
    if obj is None:
        raise ValueError("YAML is empty")
    if not isinstance(obj, (dict, list)):
        raise ValueError(f"Unexpected YAML top-level type: {type(obj).__name__}")
    return True

def lightweight_validate_zip(path: Path):
    if not zipfile.is_zipfile(path):
        raise ValueError("Not a valid zip archive")
    with zipfile.ZipFile(path, "r") as zf:
        names = [n for n in zf.namelist() if not n.endswith("/")]
        if not names:
            raise ValueError("Zip archive is empty")
    return True

for path in sorted(curated_dir.iterdir()):
    if not path.is_file() or path.name == "curation_manifest.json":
        continue
    if path.suffix.lower() in {".yaml", ".yml"}:
        try:
            lightweight_validate_yaml(path)
            shutil.copy2(path, validated_input_dir / path.name)
            validation_manifest["accepted_task1"].append(path.name)
        except Exception as e:
            validation_manifest["rejected_task1"].append({"file": path.name, "reason": repr(e)})
    elif path.suffix.lower() == ".zip":
        try:
            lightweight_validate_zip(path)
            shutil.copy2(path, validated_input_dir / path.name)
            validation_manifest["accepted_task2"].append(path.name)
        except Exception as e:
            validation_manifest["rejected_task2"].append({"file": path.name, "reason": repr(e)})

validation_manifest["accepted_total"] = len(validation_manifest["accepted_task1"]) + len(validation_manifest["accepted_task2"])
(validation_manifest_path := validated_input_dir / "validation_manifest.json").write_text(
    json.dumps(validation_manifest, ensure_ascii=False, indent=2), encoding="utf-8"
)

print("Accepted task1:", len(validation_manifest["accepted_task1"]))
print("Accepted task2:", len(validation_manifest["accepted_task2"]))
print("Rejected total:", len(validation_manifest["rejected_task1"]) + len(validation_manifest["rejected_task2"]))

if validation_manifest["accepted_total"] == 0:
    raise RuntimeError("После валидации не осталось входных файлов.")


## Проверка наличия `export-scidatapipe`


In [ ]:
# @title
help_text = subprocess.check_output(["top-papers-graph", "--help"], text=True)
print(help_text[:6000])

if "export-scidatapipe" not in help_text:
    raise RuntimeError(
        "В текущем clone нет команды export-scidatapipe. "
        "Передайте overlay-архив с patched version через REPO_OVERLAY_GDRIVE_FILE_ID/URL "
        "или используйте ветку/форк, где bridge уже добавлен."
    )


## Логин в Hugging Face

Блокнот использует `notebook_login()` или `login(token=...)`. Для загрузки результата используется `upload_folder()` в dataset repo. ([huggingface.co](https://huggingface.co/docs/huggingface_hub/guides/upload))


In [ ]:
# @title
if HF_UPLOAD:
    from huggingface_hub import login, notebook_login

    if HF_TOKEN.strip():
        print("Логин через HF_TOKEN")
        login(token=HF_TOKEN.strip(), add_to_git_credential=False)
    elif HF_USE_NOTEBOOK_LOGIN:
        print("Открываю notebook_login()")
        notebook_login()
    else:
        print("HF token не задан. Будет использован уже сохранённый токен, если он есть.")
else:
    print("HF_UPLOAD=False — логин пропущен.")


## Экспорт датасета

Если полный экспорт с `--download-papers` / multimodal enrichment падает, блокнот автоматически делает retry без этих флагов, чтобы не терять весь прогон.


In [ ]:
# @title
import shlex

output_dir = Path(OUTPUT_DIR)
if output_dir.exists():
    shutil.rmtree(output_dir)
output_dir.mkdir(parents=True, exist_ok=True)

logs_dir = output_dir / "_logs"
logs_dir.mkdir(parents=True, exist_ok=True)

def build_export_cmd(with_downloads: bool):
    cmd = [
        "top-papers-graph",
        "export-scidatapipe",
        "--input-dir", str(validated_input_dir),
        "--out-dir", str(output_dir),
        "--max-images-per-sample", str(MAX_IMAGES_PER_SAMPLE),
        "--max-multimodal-records-per-sample", str(MAX_MULTIMODAL_RECORDS_PER_SAMPLE),
    ]
    cmd += ["--copy-assets"] if COPY_ASSETS else ["--no-copy-assets"]
    cmd += ["--recursive"] if RECURSIVE else ["--no-recursive"]

    if with_downloads and DOWNLOAD_PAPERS:
        cmd += [
            "--download-papers",
            "--download-root", str(Path(DOWNLOAD_ROOT)),
            "--download-processed-papers-dir", str(Path(DOWNLOADED_PROCESSED_PAPERS_DIR)),
            "--ingest-downloaded-papers",
        ]
        if UNPAYWALL_EMAIL.strip():
            cmd += ["--download-unpaywall-email", UNPAYWALL_EMAIL.strip()]
        cmd += ["--download-multimodal"] if DOWNLOAD_MULTIMODAL else ["--no-download-multimodal"]
        cmd += ["--download-run-vlm"] if DOWNLOAD_RUN_VLM else ["--no-download-run-vlm"]

    return cmd

def run_export(label: str, with_downloads: bool):
    cmd = build_export_cmd(with_downloads=with_downloads)
    print(f"\n=== Running {label} ===")
    print(" \\\n  ".join(shlex.quote(x) for x in cmd))
    result = subprocess.run(
        cmd,
        cwd=str(repo_dir),
        text=True,
        capture_output=True,
    )
    (logs_dir / f"{label}.stdout.log").write_text(result.stdout or "", encoding="utf-8")
    (logs_dir / f"{label}.stderr.log").write_text(result.stderr or "", encoding="utf-8")
    print("Return code:", result.returncode)
    print("--- stdout tail ---")
    print((result.stdout or "")[-4000:])
    print("--- stderr tail ---")
    print((result.stderr or "")[-4000:])
    return result

result = run_export("export_with_downloads", with_downloads=True)

if result.returncode != 0:
    print("\nPrimary export failed. Retrying without download enrichment...")
    result = run_export("export_without_downloads", with_downloads=False)

if result.returncode != 0:
    raise RuntimeError(
        "Обе попытки export-scidatapipe завершились ошибкой. "
        "Смотрите логи в output/_logs."
    )

print("\nExport completed.")


## Проверка, что выход не пустой


In [ ]:
# @title
def count_jsonl_rows(path: Path):
    if not path.exists():
        return 0
    count = 0
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                count += 1
    return count

sft_path = output_dir / "sft.jsonl"
grpo_path = output_dir / "grpo.jsonl"

sft_rows = count_jsonl_rows(sft_path)
grpo_rows = count_jsonl_rows(grpo_path)

print("sft rows :", sft_rows)
print("grpo rows:", grpo_rows)

if sft_rows == 0 and grpo_rows == 0:
    raise RuntimeError(
        "Выходные JSONL пустые. Upload в Hugging Face остановлен. "
        "Смотрите output/_logs и validation_manifest.json."
    )


## Загрузка в Hugging Face

Загрузка идёт только после проверки, что JSONL не пустые. Для dataset repo используется `repo_type="dataset"` и `upload_folder()`. ([huggingface.co](https://huggingface.co/docs/huggingface_hub/guides/upload))


In [ ]:
# @title
if HF_UPLOAD:
    from huggingface_hub import HfApi, create_repo

    token = HF_TOKEN.strip() or None
    create_repo(
        repo_id=HF_REPO_ID.strip(),
        repo_type="dataset",
        private=HF_PRIVATE,
        exist_ok=True,
        token=token,
    )

    api = HfApi(token=token)
    api.upload_folder(
        folder_path=str(output_dir),
        repo_id=HF_REPO_ID.strip(),
        repo_type="dataset",
        path_in_repo=HF_PATH_IN_REPO.strip() or ".",
        commit_message=HF_COMMIT_MESSAGE,
        commit_description=HF_COMMIT_DESCRIPTION,
        ignore_patterns=["_logs/*"],
    )
    print("Upload completed.")
else:
    print("HF_UPLOAD=False — upload skipped.")


## Просмотр результата


In [ ]:
# @title
print("=== curation_manifest.json ===")
curation_manifest_path = curated_dir / "curation_manifest.json"
if curation_manifest_path.exists():
    print(curation_manifest_path.read_text(encoding="utf-8")[:20000])

print("\n=== validation_manifest.json ===")
if validation_manifest_path.exists():
    print(validation_manifest_path.read_text(encoding="utf-8")[:20000])

print("\n=== Output files ===")
for p in sorted(output_dir.rglob("*")):
    if p.is_file():
        rel = p.relative_to(output_dir)
        print("-", rel, f"({p.stat().st_size/1024:.1f} KB)")

stats_path = output_dir / "stats.json"
if stats_path.exists():
    print("\n=== stats.json ===")
    print(stats_path.read_text(encoding="utf-8"))

for candidate in ["sft.jsonl", "grpo.jsonl"]:
    path = output_dir / candidate
    if path.exists():
        print(f"\n=== First lines from {candidate} ===")
        with path.open("r", encoding="utf-8") as f:
            for i, line in enumerate(f):
                print(line[:1500])
                if i >= 1:
                    break
